# Convex Optimization — Complete Ground-Up Guide
### For quant developer interviews | CS background, maths built from scratch

---

**What this notebook covers:**

1. What is convexity? Geometric and algebraic definitions with finance examples
2. Why convexity matters: the global optimum guarantee vs non-convex problems
3. Taxonomy of convex problems: LP, QP, QCQP, SOCP — and where each arises in finance
4. KKT conditions: the mathematics of constrained optimality
5. Lagrangian duality — shadow prices and economic interpretation
6. cvxpy deep dive: DCP rules, Parameters, warm-starting, solver selection
7. Mean-variance optimization as a QP — full problem formulation
8. Rebalancing with transaction costs as a QP+L1 problem
9. CVaR minimization as a Linear Program (Rockafellar-Uryasev 2000)
10. Production patterns: compilation, batching, fallback, robustness

**Dependencies:**
```
pip install numpy scipy matplotlib cvxpy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import minimize
from scipy.stats import t as t_dist
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import cvxpy as cp
    CVXPY = True
    print(f'cvxpy {cp.__version__} — OK')
except ImportError:
    CVXPY = False
    print('cvxpy not found: pip install cvxpy')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f8f8',
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 0.8,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
})

PURPLE = '#534AB7'; TEAL = '#1D9E75'; AMBER = '#EF9F27'
CORAL  = '#D85A30'; BLUE = '#185FA5'; GREEN = '#3B6D11'
RED    = '#A32D2D'; GRAY = '#888780'

np.random.seed(42)
print('Imports complete.')

---
## Part 1 — What Is Convexity?

### Convex sets
A set $\mathcal{C} \subseteq \mathbb{R}^n$ is **convex** if for any two points $x, y \in \mathcal{C}$, the entire line segment between them lies in $\mathcal{C}$:
$$\forall\, x, y \in \mathcal{C},\; \theta \in [0,1]: \quad \theta x + (1-\theta)y \in \mathcal{C}$$

Examples you must know: hyperplanes, half-spaces, balls, the positive orthant $\{x: x \geq 0\}$, polyhedra (intersections of half-spaces). The feasible set of a portfolio problem — weights summing to 1, bounded between 0 and some cap — is a convex polytope (intersection of half-spaces and a hyperplane).

### Convex functions
A function $f: \mathbb{R}^n \to \mathbb{R}$ is **convex** if its domain is convex and:
$$f(\theta x + (1-\theta)y) \leq \theta f(x) + (1-\theta)f(y) \quad \forall x, y,\; \theta\in[0,1]$$

**Geometrically:** the function lies **below or on the chord** connecting any two points. For twice-differentiable functions, this is equivalent to the **Hessian being positive semi-definite** everywhere: $\nabla^2 f \succeq 0$.

### Operations that preserve convexity
These are the rules cvxpy uses internally to certify problems:
- **Non-negative linear combination:** $f = \alpha_1 f_1 + \alpha_2 f_2$, $\alpha_i \geq 0$ — convex
- **Pointwise maximum:** $f(x) = \max(f_1(x), f_2(x))$ — convex. This means $|x| = \max(x, -x)$ is convex
- **Composition with affine map:** $f(Ax + b)$ is convex if $f$ is convex
- **Inf-convolution, perspective** — advanced but preserve convexity

### The central theorem of convex optimization
> For a convex problem (convex objective + convex feasible set), **every local minimum is the unique global minimum.**

This is why convex optimization is so powerful in production finance. Non-convex problems (neural networks, combinatorial assignments) can have exponentially many local traps. A convex problem has exactly one basin — any descent algorithm converges to the global answer.

In [ ]:
# ── Visualise convexity: sets, functions, local=global ──────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

# 1. Convex vs non-convex set
ax = axes[0]
theta_c = np.linspace(0, 2*np.pi, 200)
ax.fill(np.cos(theta_c), np.sin(theta_c), color=PURPLE, alpha=0.2, label='Convex (ball)')
ax.plot(np.cos(theta_c), np.sin(theta_c), color=PURPLE, lw=2)
p1, p2 = np.array([-0.7, -0.6]), np.array([0.6, 0.5])
ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color=TEAL, lw=2.5, label='Chord inside set')
ax.scatter(*p1, s=60, color=TEAL, zorder=5)
ax.scatter(*p2, s=60, color=TEAL, zorder=5)

# Non-convex: crescent (offset to the right)
theta2 = np.linspace(0.4, np.pi*1.6, 100)
cx = np.concatenate([np.cos(theta2)+3.5, 0.55*np.cos(theta2[::-1])+3.5])
cy = np.concatenate([np.sin(theta2),     0.55*np.sin(theta2[::-1])])
ax.fill(cx, cy, color=CORAL, alpha=0.2, label='Non-convex (crescent)')
ax.plot(cx, cy, color=CORAL, lw=2)
q1, q2 = np.array([2.6, 0.8]), np.array([4.4, 0.8])
ax.plot([q1[0], q2[0]], [q1[1], q2[1]], color=RED, lw=2.5, ls='--', label='Chord exits set')
ax.scatter(*q1, s=60, color=RED, zorder=5)
ax.scatter(*q2, s=60, color=RED, zorder=5)

ax.set_xlim(-1.5, 5.2); ax.set_ylim(-1.5, 1.7)
ax.set_aspect('equal')
ax.set_title('Convex set (left) vs\nnon-convex set (right)')
ax.legend(fontsize=7, loc='lower left')
ax.grid(False); ax.set_facecolor('white')
ax.text(0, -1.3, 'CONVEX', ha='center', color=PURPLE, fontweight='bold', fontsize=9)
ax.text(3.8, -1.3, 'NON-CONVEX', ha='center', color=CORAL, fontweight='bold', fontsize=9)

# 2. Convex function f(x) = x²
ax = axes[1]
x = np.linspace(-3, 3, 200)
ax.plot(x, x**2, color=PURPLE, lw=2.5, label='f(x) = x² (convex)')
x1, x2 = -2.0, 1.5
ax.plot([x1, x2], [x1**2, x2**2], color=AMBER, lw=2, ls='--', label='Chord (above f)')
for theta in [0.25, 0.5, 0.75]:
    xm = theta*x1 + (1-theta)*x2
    ax.scatter([xm], [xm**2], s=50, color=TEAL, zorder=5)
    ax.scatter([xm], [theta*x1**2 + (1-theta)*x2**2], s=50, color=AMBER, marker='^', zorder=5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Convex function: chord always above f\nHessian ∇²f = 2 ≥ 0 everywhere')
ax.legend(fontsize=8)
ax.annotate('Function (teal dots)', xy=(0.5, 0.25), xytext=(-1.5, 5),
            fontsize=8, color=TEAL, arrowprops=dict(arrowstyle='->', color=TEAL))

# 3. Non-convex: multiple local minima
ax = axes[2]
f_nc = lambda x: np.sin(2.5*x) + 0.3*x**2 - x
ax.plot(x, f_nc(x), color=CORAL, lw=2.5, label='Non-convex function')
from scipy.optimize import minimize_scalar
local_x = [-2.0, 0.4, 1.9]
for i, lx in enumerate(local_x):
    res = minimize_scalar(f_nc, bounds=(lx-0.5, lx+0.5), method='bounded')
    col = GREEN if i == 1 else RED
    label = 'Global minimum' if i == 1 else 'Local minimum'
    ax.scatter([res.x], [res.fun], s=80, color=col, zorder=5)
    ax.annotate(label, (res.x, res.fun), textcoords='offset points',
                xytext=(8, 5), fontsize=8, color=col)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Non-convex: multiple local minima\nGradient descent gets stuck!')
ax.legend(fontsize=8)

# 4. Key convex functions in finance
ax = axes[3]
xr = np.linspace(-2.5, 3.5, 300)
ax.plot(xr, xr**2,                color=PURPLE, lw=2,   label='x²  (variance / QP objective)')
ax.plot(xr, np.abs(xr),           color=TEAL,   lw=2,   label='|x|  (L1 transaction cost)')
ax.plot(xr, np.maximum(xr, 0),    color=AMBER,  lw=2,   label='max(x,0)  (tax on gains)')
ax.plot(xr, np.exp(np.clip(xr,-2,2))/4, color=CORAL, lw=2, label='exp(x)/4  (penalty / barrier)')
ax.axhline(0, color='black', lw=0.8)
ax.set_ylim(-0.3, 4); ax.set_xlim(-2.5, 3.5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Convex functions used in finance\nAll valid as cvxpy objective/constraint terms')
ax.legend(fontsize=7)

plt.suptitle('Convexity: geometric intuition and finance relevance', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

# Print convexity classification for common finance expressions
print('Convexity classification of common finance expressions:')
checks = [
    ("w'Σw  (portfolio variance)",          "CONVEX  — quadratic form, Σ ≽ 0 (PSD)"),
    ("Σ|Δwᵢ|  (L1 transaction costs)",      "CONVEX  — sum of |·|, each convex"),
    ("max(0, gain·trade)  (tax on sells)",   "CONVEX  — cp.pos(x), nonneg scaling"),
    ("CVaR (expected shortfall)",            "CONVEX  — Rockafellar-Uryasev LP formulation"),
    ("w'μ  (expected return, maximise)",     "LINEAR  — negate to minimise → convex obj"),
    ("σ_p / (R_p - rf)  (Sharpe ratio)",    "NON-CONVEX — ratio; needs SOCP reformulation"),
    ("max drawdown",                         "NON-CONVEX — path-dependent, no known convex form"),
    ("w'μ - λ/2 · w'Σw  (MV utility)",      "CONCAVE — maximise ⟺ minimise negative"),
]
for name, verdict in checks:
    symbol = 'OK' if 'CONVEX' in verdict or 'LINEAR' in verdict or 'CONCAVE' in verdict else 'XX'
    print(f'  [{symbol}] {name}')
    print(f'        {verdict}')

---
## Part 2 — Taxonomy of Convex Problems

All convex optimization problems have the form:
$$\underset{x}{\text{minimise}}\; f_0(x) \quad \text{s.t.}\; f_i(x) \leq 0,\; h_j(x) = 0$$
where all $f_i$ are convex and all $h_j$ are affine.

### LP — Linear Program
$$\min\; c^\top x \quad \text{s.t.}\; Ax \leq b,\; Cx = d,\; x \geq 0$$
Linear objective, linear constraints. Solved in polynomial time. In finance: CVaR minimisation, minimum-cost index replication, LP-relaxations of discrete problems.

### QP — Quadratic Program
$$\min\; \tfrac{1}{2}x^\top Qx + c^\top x \quad \text{s.t.}\; Ax \leq b,\; Cx = d$$
$Q \succeq 0$ makes it convex. **Portfolio variance minimisation is a QP.** Transaction cost rebalancing with L1 costs is also a QP (via variable splitting: $|\Delta w| = u + v$, $\Delta w = u - v$, $u,v \geq 0$).

### QCQP — Quadratically Constrained QP
QP with additional quadratic inequality constraints: $x^\top Q_i x + c_i^\top x \leq b_i$. Portfolio volatility constraints ($\sigma_p \leq \bar{\sigma}$ → $w^\top\Sigma w \leq \bar{\sigma}^2$) turn MVO into a QCQP.

### SOCP — Second-Order Cone Program
Constraint: $\|Ax + b\|_2 \leq c^\top x + d$. More general than QP. Robust MVO with parameter uncertainty, Sharpe ratio maximisation (after reformulation), and Markowitz with uncertain covariance all become SOCPs.

### SDP — Semidefinite Program
Matrix variable $X \succeq 0$. Covariance matrix estimation with PSD constraints, certain factor model regularisation. Rarely needed directly in robo-advisor applications.

**Solver matching rule:**

| Problem class | Best solver (via cvxpy) | Why |
|---|---|---|
| LP | HiGHS, GLPK | Simplex or interior-point, highly optimised |
| QP | OSQP, Clarabel | Operator splitting, extremely fast for sparse QPs |
| SOCP | ECOS, Clarabel | Second-order cone interior-point |
| SDP | SCS, MOSEK | First-order or interior-point for large SDPs |
| Large-scale | SCS | First-order, handles millions of variables, less accurate |

In [ ]:
# ── LP, QP, SOCP examples in finance context ────────────────────────────────
if not CVXPY:
    print('cvxpy required for this section')
else:
    N = 4
    asset_names = ['Global Eq', 'EM Eq', 'Gov Bond', 'Corp Bond']
    mu    = np.array([0.10, 0.14, 0.04, 0.09])
    vols  = np.array([0.18, 0.22, 0.06, 0.12])
    corr  = np.array([[ 1.00,  0.65, -0.20,  0.35],
                      [ 0.65,  1.00, -0.10,  0.30],
                      [-0.20, -0.10,  1.00,  0.50],
                      [ 0.35,  0.30,  0.50,  1.00]])
    Sigma = np.outer(vols, vols) * corr
    rf    = 0.025

    results_probs = {}

    # --- LP: minimum transaction cost to achieve return target ---------------
    w_lp     = cp.Variable(N)
    cost_lp  = np.array([0.0003, 0.0010, 0.0001, 0.0004])
    w_lp_start = np.array([0.40, 0.20, 0.20, 0.20])
    prob_lp  = cp.Problem(
        cp.Minimize(cost_lp @ cp.abs(w_lp - w_lp_start)),
        [cp.sum(w_lp) == 1, w_lp >= 0, w_lp @ mu >= 0.09]
    )
    prob_lp.solve(verbose=False)
    results_probs['LP: min cost\n(return >= 9%)'] = {
        'w': w_lp.value, 'color': BLUE,
        'ret': float(w_lp.value @ mu), 'vol': float(np.sqrt(w_lp.value @ Sigma @ w_lp.value))
    }

    # --- QP: minimum variance portfolio -------------------------------------
    w_qp = cp.Variable(N)
    prob_qp = cp.Problem(
        cp.Minimize(cp.quad_form(w_qp, Sigma)),
        [cp.sum(w_qp) == 1, w_qp >= 0]
    )
    prob_qp.solve(verbose=False)
    results_probs['QP: min variance\n(long-only)'] = {
        'w': w_qp.value, 'color': PURPLE,
        'ret': float(w_qp.value @ mu), 'vol': float(np.sqrt(w_qp.value @ Sigma @ w_qp.value))
    }

    # --- QP + L1: rebalancing with transaction costs -----------------------
    w_curr = np.array([0.52, 0.18, 0.10, 0.20])
    w_tgt  = np.array([0.30, 0.25, 0.25, 0.20])
    cost_r = np.array([0.0003, 0.0010, 0.0001, 0.0004])
    w_reb  = cp.Variable(N)
    prob_reb = cp.Problem(
        cp.Minimize(
            10 * cp.quad_form(w_reb - w_tgt, Sigma)
            + cost_r @ cp.abs(w_reb - w_curr)
        ),
        [cp.sum(w_reb) == 1, w_reb >= 0,
         cp.sum(cp.abs(w_reb - w_curr)) <= 0.30]
    )
    prob_reb.solve(verbose=False)
    results_probs['QP+L1: rebalancing\n(TE + costs)'] = {
        'w': w_reb.value, 'color': TEAL,
        'ret': float(w_reb.value @ mu), 'vol': float(np.sqrt(w_reb.value @ Sigma @ w_reb.value))
    }

    # --- QCQP: max return with hard volatility cap (SOCP internally) -------
    w_sq   = cp.Variable(N)
    sig_max = 0.10
    prob_sq = cp.Problem(
        cp.Maximize(w_sq @ mu),
        [cp.sum(w_sq) == 1, w_sq >= 0,
         cp.quad_form(w_sq, Sigma) <= sig_max**2]
    )
    prob_sq.solve(verbose=False)
    results_probs['QCQP: max return\n(vol <= 10%)'] = {
        'w': prob_sq.variables()[0].value, 'color': AMBER,
        'ret': float(prob_sq.variables()[0].value @ mu),
        'vol': float(np.sqrt(prob_sq.variables()[0].value @ Sigma @ prob_sq.variables()[0].value))
    }

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Weights
    x = np.arange(N)
    wb = 0.20
    for i, (pname, rd) in enumerate(results_probs.items()):
        if rd['w'] is not None:
            axes[0].bar(x + (i - 1.5)*wb, rd['w']*100, wb,
                        color=rd['color'], label=pname.replace('\n', ' '), alpha=0.85, edgecolor='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(asset_names)
    axes[0].set_ylabel('Weight (%)')
    axes[0].set_title('Optimal weights by problem type\nSame assets, different objectives -> different portfolios')
    axes[0].legend(fontsize=7)

    # Risk-return with efficient frontier
    from scipy.optimize import minimize as sp_min
    target_rs = np.linspace(mu.min()+0.001, mu.max()-0.001, 60)
    fv, fr = [], []
    for tr in target_rs:
        res_f = sp_min(lambda w: w @ Sigma @ w, np.ones(N)/N, method='SLSQP',
                       bounds=[(0,1)]*N,
                       constraints=[{'type':'eq','fun':lambda w:w.sum()-1},
                                    {'type':'eq','fun':lambda w,t=tr: w@mu-t}])
        if res_f.success:
            fv.append(np.sqrt(res_f.fun)*100); fr.append(tr*100)
    axes[1].plot(fv, fr, color=GRAY, lw=2, alpha=0.5, label='Efficient frontier', zorder=1)

    for pname, rd in results_probs.items():
        if rd['w'] is not None:
            axes[1].scatter([rd['vol']*100], [rd['ret']*100], s=120,
                            color=rd['color'], zorder=5)
            axes[1].annotate(pname.replace('\n', ' '),
                             (rd['vol']*100, rd['ret']*100),
                             textcoords='offset points', xytext=(6, 3), fontsize=8, color=rd['color'])
    axes[1].scatter([0], [rf*100], s=80, color=GRAY, marker='D', zorder=5, label=f'Risk-free ({rf*100:.1f}%)')
    axes[1].set_xlabel('Annual volatility (%)')
    axes[1].set_ylabel('Expected return (%)')
    axes[1].set_title('All solutions on or near the efficient frontier')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print('Problem summary:')
    print(f'  {"Problem":<35} {"Return":>8} {"Vol":>8} {"Status":<15}')
    print('-' * 72)
    for pname, rd in results_probs.items():
        print(f'  {pname.replace(chr(10)," "):<35} {rd["ret"]*100:>7.2f}% {rd["vol"]*100:>7.2f}%')

---
## Part 3 — KKT Conditions: The Mathematics of Optimality

The **Karush-Kuhn-Tucker (KKT) conditions** are necessary and sufficient for optimality in a convex problem with constraint qualification. They give you the mathematical fingerprint of an optimal solution — and their economic interpretation is essential for understanding portfolio optimization.

### The Lagrangian
For the general problem $\min f_0(x)$ s.t. $f_i(x) \leq 0$, $h_j(x) = 0$:
$$\mathcal{L}(x, \lambda, \nu) = f_0(x) + \sum_i \lambda_i f_i(x) + \sum_j \nu_j h_j(x)$$

$\lambda_i \geq 0$ are the **Lagrange multipliers** (dual variables) for inequality constraints. They have a critical economic interpretation: $\lambda_i$ is the **shadow price** of constraint $i$ — the rate at which the optimal objective improves per unit of constraint relaxation.

### The four KKT conditions

1. **Primal feasibility:** $f_i(x^*) \leq 0$, $h_j(x^*) = 0$ — the solution is feasible
2. **Dual feasibility:** $\lambda_i \geq 0$ — shadow prices are non-negative for inequality constraints
3. **Stationarity:** $\nabla_{x}\mathcal{L} = 0$ — the Lagrangian gradient vanishes at the optimal point
4. **Complementary slackness:** $\lambda_i \cdot f_i(x^*) = 0\;\forall i$ — **the most important one**

### Complementary slackness — deep interpretation
Either the constraint is **binding** ($f_i(x^*) = 0$) **OR** the shadow price is **zero** ($\lambda_i = 0$). Never both positive simultaneously.

**Portfolio interpretation:** For long-only constraint $-w_i \leq 0$:
- Asset $i$ is **held** ($w_i > 0$): the constraint is not binding → $\lambda_i = 0$ (no shadow price)
- Asset $i$ is **excluded** ($w_i = 0$): the constraint is binding → $\lambda_i > 0$ (there is value in being long, but it doesn't outweigh its cost)

For the budget constraint $\sum w_i = 1$ (equality): the Lagrange multiplier $\nu$ is the **marginal value of an additional unit of investable capital** — literally the risk-adjusted return of the marginal investment.

### Stationarity for MVO — the closed-form solution
For min-variance with just the budget constraint:
$$\nabla_w(w^\top\Sigma w) + \nu\cdot\mathbf{1} = 0$$
$$2\Sigma w = -\nu\cdot\mathbf{1} \implies w^* = -\frac{\nu}{2}\Sigma^{-1}\mathbf{1}$$
Normalizing so $\mathbf{1}^\top w = 1$: $w^* = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}$

This is the **global minimum variance portfolio** — the closed-form solution that exists only without inequality constraints. With long-only constraints, the KKT system must be solved numerically.

In [ ]:
# ── KKT conditions: solve QP, inspect dual variables, verify optimality ─────
if not CVXPY:
    print('cvxpy required')
else:
    # Minimum variance portfolio with several constraints
    w_kkt = cp.Variable(N)
    constr_kkt = [
        cp.sum(w_kkt) == 1,         # equality: budget
        w_kkt >= 0,                  # inequality: long-only
        w_kkt <= 0.50,               # inequality: concentration cap
        w_kkt @ mu >= 0.075,         # inequality: minimum return 7.5%
    ]
    prob_kkt = cp.Problem(cp.Minimize(cp.quad_form(w_kkt, Sigma)), constr_kkt)
    prob_kkt.solve(verbose=False)

    w_opt    = w_kkt.value
    nu_budget = float(constr_kkt[0].dual_value)      # scalar
    lam_long  = constr_kkt[1].dual_value              # (N,) vector
    lam_cap   = constr_kkt[2].dual_value              # (N,) vector
    lam_ret   = float(constr_kkt[3].dual_value)       # scalar

    # Verify stationarity: 2Σw + ν·1 - λ_long + λ_cap + λ_ret·μ = 0
    grad_obj  = 2 * Sigma @ w_opt
    kkt_resid = grad_obj + nu_budget*np.ones(N) - lam_long + lam_cap + lam_ret*mu
    max_resid = np.abs(kkt_resid).max()

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Optimal weights
    colors_w = [GREEN if w > 1e-4 else GRAY for w in w_opt]
    bars = axes[0].bar(asset_names, w_opt*100, color=colors_w, edgecolor='white', width=0.55)
    for bar, w_val in zip(bars, w_opt):
        if w_val > 1e-4:
            axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                         f'{w_val*100:.1f}%', ha='center', fontsize=10, fontweight='bold')
    axes[0].set_ylabel('Weight (%)')
    axes[0].set_title(f'Min-variance portfolio\n(return >= 7.5%, long-only, cap <= 50%)')
    pv = np.sqrt(w_opt @ Sigma @ w_opt) * 100
    axes[0].text(0.97, 0.97, f'Portfolio vol:\n{pv:.1f}%/year',
                 transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    # Dual variables — shadow prices
    x_kkt = np.arange(N)
    w_d = 0.35
    axes[1].bar(x_kkt - w_d/2, lam_long, w_d, color=CORAL,  label='lambda_long (long constraint)', alpha=0.85)
    axes[1].bar(x_kkt + w_d/2, lam_cap,  w_d, color=BLUE,   label='lambda_cap  (cap constraint)',  alpha=0.85)
    axes[1].set_xticks(x_kkt); axes[1].set_xticklabels(asset_names)
    axes[1].axhline(0, color='black', lw=1)
    axes[1].set_ylabel('Dual variable (shadow price)')
    axes[1].set_title('KKT dual variables (shadow prices)\nNon-zero only when constraint is binding')
    axes[1].legend(fontsize=8)

    # Stationarity residual and complementary slackness
    cs_long = np.abs(w_opt * lam_long)
    cs_cap  = np.abs((0.50 - w_opt) * lam_cap)
    x_cs = np.arange(N); w_cs = 0.35
    axes[2].bar(x_cs - w_cs/2, cs_long, w_cs, color=CORAL, label='|w * lambda_long|', alpha=0.85)
    axes[2].bar(x_cs + w_cs/2, cs_cap,  w_cs, color=BLUE,  label='|(0.5-w)*lambda_cap|', alpha=0.85)
    axes[2].set_xticks(x_cs); axes[2].set_xticklabels(asset_names)
    axes[2].set_ylabel('Complementary slackness residual')
    axes[2].set_title(f'Complementary slackness: should be ~0\nStationarity max residual: {max_resid:.2e}')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print('KKT conditions verification:')
    print(f'  Solver status:                  {prob_kkt.status}')
    print(f'  Stationarity max residual:      {max_resid:.2e}  (< 1e-6 => verified)')
    print(f'  Dual feasibility (lambda >= 0): {(lam_long >= -1e-6).all() and (lam_cap >= -1e-6).all()}')
    print(f'  Compl. slackness max:           {cs_long.max():.2e}')
    print()
    print(f'Shadow price interpretation:')
    print(f'  Budget multiplier nu = {nu_budget:.4f}  (marginal value of 1 extra unit of capital)')
    print(f'  Return constraint lambda = {lam_ret:.5f}  (relaxing return >= 7.5% to 7.4% saves {lam_ret*0.001:.6f} var)')
    for i, name in enumerate(asset_names):
        if lam_long[i] > 1e-5:
            print(f'  Long constraint on {name}: BINDING (w=0), lambda={lam_long[i]:.4f}')
        else:
            print(f'  Long constraint on {name}: not binding (w={w_opt[i]*100:.1f}%), lambda~0')

---
## Part 4 — cvxpy Deep Dive: DCP Rules, Parameters, Warm-Starting

cvxpy is the production library for convex optimization in Python. At Scalable Capital you will write production cvxpy code that runs for hundreds of thousands of clients. Understanding the internals is essential.

### DCP — Disciplined Convex Programming
cvxpy verifies problem convexity by checking every expression against a set of **composition rules**:
- A **convex** expression can appear in `minimize()` or in a `<=` constraint
- A **concave** expression can appear in `maximize()` or in a `>=` constraint
- An **affine** expression can appear anywhere

If your problem fails DCP, cvxpy raises `DCPError`. This does not always mean the problem is non-convex — sometimes you need to **reformulate**:
- `cp.abs(x)` — recognized as convex
- `cp.pos(x)` — `max(x, 0)`, recognized as convex
- `cp.norm(x)` — Euclidean norm, convex
- `cp.quad_form(x, Q)` — $x^\top Qx$, convex if `Q` is PSD
- `cp.sum_squares(x)` — $\|x\|_2^2$, always convex
- `cp.maximum(a, b)` — pointwise max, convex

### Parameters — the key to production performance
A `cp.Parameter` is a placeholder whose **value** can change between solves without recompiling the problem graph. The problem graph compilation (parsing, DCP check, canonicalization, sparsity analysis) costs ~10ms. The actual solve costs ~1–50ms. If you recompile every solve, you waste 10ms per client — at 300K clients, that's 50 minutes just in compilation overhead.

```python
# WRONG for production: recompiles every call
def solve(Sigma, w_curr, w_tgt):
    w = cp.Variable(N)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w - w_tgt, Sigma)), ...)
    prob.solve()

# RIGHT: compile once, update parameters per client
Sigma_p  = cp.Parameter((N, N), PSD=True)
w_curr_p = cp.Parameter(N)
w_tgt_p  = cp.Parameter(N)
w        = cp.Variable(N)
prob     = cp.Problem(cp.Minimize(cp.quad_form(w - w_tgt_p, Sigma_p)), ...)

def solve(Sigma, w_curr, w_tgt):
    Sigma_p.value  = Sigma
    w_curr_p.value = w_curr
    w_tgt_p.value  = w_tgt
    prob.solve(warm_start=True)
    return w.value
```

### Warm-starting
Pass `warm_start=True` and the solver starts from the previous solution. For sequential clients with similar portfolios (same strategy, slightly different weights), this reduces solver iterations by 2–5x.

### Solver choice guide

| Solver | Best for | Notes |
|---|---|---|
| **OSQP** | QP (quadratic obj, linear constraints) | Default for rebalancing; sparse, fast, first-order |
| **Clarabel** | QP and SOCP | New default in cvxpy 1.3+; often fastest |
| **ECOS** | SOCP, small-medium problems | Accurate interior-point; good for CVaR LP |
| **SCS** | Large-scale, first-order | Less accurate but handles huge problems |
| **GLPK/HiGHS** | LP only | Very fast for pure LPs |

In [ ]:
# ── cvxpy: DCP rules, Parameters benchmark, warm-starting, solver comparison ─
if not CVXPY:
    print('cvxpy required')
else:
    # ── 1. DCP compliance checks ─────────────────────────────────────────────
    print('=== DCP compliance checks ===')
    x_dcp = cp.Variable(3)
    Q_dcp = np.eye(3)
    A_dcp = np.random.randn(3, 3)

    dcp_tests = [
        ('cp.sum_squares(x)',       cp.sum_squares(x_dcp),            True),
        ('cp.abs(x[0])',            cp.abs(x_dcp[0]),                  True),
        ('cp.pos(x[0])',            cp.pos(x_dcp[0]),                  True),
        ('cp.quad_form(x, I)',      cp.quad_form(x_dcp, Q_dcp),        True),
        ('cp.norm(x)',              cp.norm(x_dcp),                    True),
        ('-cp.sum_squares(x)',      -cp.sum_squares(x_dcp),            False),  # concave
        ('cp.log(cp.sum_exp(x))',   cp.log_sum_exp(x_dcp),            True),
    ]
    for name_d, expr, should_convex in dcp_tests:
        is_c = expr.is_convex()
        is_cc = expr.is_concave()
        tag = 'CONVEX' if is_c else ('CONCAVE' if is_cc else 'NEITHER')
        print(f'  {name_d:<30}: {tag}')

    print()

    # ── 2. Parameters vs recompilation: timing benchmark ────────────────────
    print('=== Parameter vs recompilation benchmark ===')
    N_b  = 8
    n_s  = 40

    def make_spd(n):
        A = np.random.randn(n, n)
        return A @ A.T / n + np.eye(n) * 0.01

    def rand_weights(n):
        w = np.abs(np.random.randn(n))
        return w / w.sum()

    test_data = [(make_spd(N_b), rand_weights(N_b), rand_weights(N_b)) for _ in range(n_s)]

    # Method A: recompile
    t0 = time.time()
    for S, wc, wt in test_data:
        w_tmp = cp.Variable(N_b)
        p_tmp = cp.Problem(
            cp.Minimize(10*cp.quad_form(w_tmp - wt, S) + 0.001*cp.sum(cp.abs(w_tmp - wc))),
            [cp.sum(w_tmp)==1, w_tmp>=0]
        )
        p_tmp.solve(solver=cp.OSQP, verbose=False)
    t_recomp = time.time() - t0

    # Method B: parameters
    Sp_b  = cp.Parameter((N_b, N_b), PSD=True)
    wcp_b = cp.Parameter(N_b)
    wtp_b = cp.Parameter(N_b)
    wp_b  = cp.Variable(N_b)
    prob_b = cp.Problem(
        cp.Minimize(10*cp.quad_form(wp_b - wtp_b, Sp_b) + 0.001*cp.sum(cp.abs(wp_b - wcp_b))),
        [cp.sum(wp_b)==1, wp_b>=0]
    )
    t0 = time.time()
    for S, wc, wt in test_data:
        Sp_b.value = S; wcp_b.value = wc; wtp_b.value = wt
        prob_b.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    t_param = time.time() - t0

    print(f'  {n_s} solves, N={N_b} assets:')
    print(f'  Recompile each time : {t_recomp:.3f}s  ({t_recomp/n_s*1000:.1f} ms/solve)')
    print(f'  Parameters + warmstart: {t_param:.3f}s  ({t_param/n_s*1000:.1f} ms/solve)')
    print(f'  Speedup: {t_recomp/t_param:.1f}x')
    print()

    # ── 3. Solver comparison on a QP ─────────────────────────────────────────
    print('=== Solver comparison on min-variance QP ===')
    S_t, wc_t, wt_t = test_data[0]
    solver_names = ['OSQP', 'ECOS', 'SCS']
    if hasattr(cp, 'CLARABEL'):
        solver_names.insert(0, 'CLARABEL')

    solver_res = {}
    for sname in solver_names:
        try:
            w_s  = cp.Variable(N_b)
            p_s  = cp.Problem(cp.Minimize(cp.quad_form(w_s, S_t)), [cp.sum(w_s)==1, w_s>=0])
            t0   = time.time()
            p_s.solve(solver=getattr(cp, sname), verbose=False)
            dt   = time.time() - t0
            solver_res[sname] = {'ms': dt*1000, 'status': p_s.status, 'val': p_s.value}
            print(f'  {sname:<12}: {dt*1000:.2f} ms  |  status: {p_s.status:<20}  |  obj: {p_s.value:.6f}')
        except Exception as e:
            print(f'  {sname:<12}: {e}')

    # ── 4. Visualise parameter speedup and solver comparison ─────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Speedup bar
    axes[0].bar(['Recompile\neach solve', 'Parameters\n+ warm start'],
                [t_recomp/n_s*1000, t_param/n_s*1000],
                color=[CORAL, TEAL], edgecolor='white', width=0.5)
    axes[0].set_ylabel('Time per solve (ms)')
    axes[0].set_title(f'cvxpy parameter speedup\n{n_s} solves, N={N_b} assets')
    for i, val in enumerate([t_recomp/n_s*1000, t_param/n_s*1000]):
        axes[0].text(i, val + 0.2, f'{val:.1f} ms', ha='center', fontsize=11, fontweight='bold')

    # Solver times
    if solver_res:
        snames = list(solver_res.keys())
        stimes = [solver_res[s]['ms'] for s in snames]
        bar_cols = [BLUE, PURPLE, TEAL, AMBER, GREEN][:len(snames)]
        axes[1].bar(snames, stimes, color=bar_cols, edgecolor='white', width=0.55)
        axes[1].set_ylabel('Solve time (ms)')
        axes[1].set_title(f'Solver comparison on N={N_b} min-variance QP')
        for i, (s, t_v) in enumerate(zip(snames, stimes)):
            axes[1].text(i, t_v + 0.02, f'{t_v:.2f}ms', ha='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()

---
## Part 5 — Mean-Variance Optimization: Full QP Formulation

This is the core problem every Scalable Capital quant developer must know cold. Here is the complete problem, every constraint explained, and the cvxpy implementation.

### Standard form
$$\underset{\mathbf{w}}{\text{minimise}} \quad \underbrace{\mathbf{w}^\top\Sigma\mathbf{w}}_{\text{portfolio variance}} - \frac{1}{\lambda}\,\mathbf{w}^\top\boldsymbol{\mu}$$

$$\text{subject to:} \quad \mathbf{1}^\top\mathbf{w} = 1 \quad \text{(fully invested)}$$
$$w_i \geq 0 \quad \forall i \quad \text{(long-only)}$$
$$w_i \leq c_i \quad \forall i \quad \text{(concentration caps)}$$
$$\mathbf{g}^\top\mathbf{w} \geq \bar{g} \quad \text{(sector/factor constraints)}$$

### Why is it a QP?
- Objective: $\mathbf{w}^\top\Sigma\mathbf{w}$ is quadratic in $\mathbf{w}$, linear term $\mathbf{w}^\top\boldsymbol{\mu}$ is linear → **quadratic objective**
- All constraints are linear in $\mathbf{w}$ → **linear constraints**
- $\Sigma \succeq 0$ → convex quadratic objective

This is a textbook **convex QP**. Solvers: OSQP (operator splitting), Clarabel (interior-point).

### Efficient frontier: sweep $\lambda$
By solving for many values of $\lambda \in (0, \infty)$:
- $\lambda \to \infty$: infinitely risk-averse → minimum variance portfolio
- $\lambda \to 0$: risk-neutral → maximum return portfolio  
- Intermediate: the entire efficient frontier

In a robo-advisor, each **risk score** (e.g., 1–10) maps to a specific $\lambda$ value. The mapping is calibrated so that the resulting portfolio volatility matches the stated risk target for that score level.

In [ ]:
# ── Full MVO implementation with efficient frontier sweep ────────────────────
if not CVXPY:
    print('cvxpy required')
else:
    N_mv = 5
    asset_names_mv = ['MSCI World', 'EM Equity', 'US Bonds', 'EU Bonds', 'Commodities']
    mu_mv   = np.array([0.09, 0.12, 0.035, 0.028, 0.06])
    vols_mv = np.array([0.17, 0.21, 0.055, 0.048, 0.18])
    corr_mv = np.array([
        [ 1.00,  0.72, -0.25, -0.20,  0.15],
        [ 0.72,  1.00, -0.18, -0.12,  0.10],
        [-0.25, -0.18,  1.00,  0.80, -0.05],
        [-0.20, -0.12,  0.80,  1.00, -0.08],
        [ 0.15,  0.10, -0.05, -0.08,  1.00],
    ])
    Sigma_mv = np.outer(vols_mv, vols_mv) * corr_mv
    rf_mv    = 0.025

    # ── Compile MVO template with parameters ─────────────────────────────────
    mu_p    = cp.Parameter(N_mv)
    Sigma_p = cp.Parameter((N_mv, N_mv), PSD=True)
    lam_p   = cp.Parameter(nonneg=True)

    w_mv = cp.Variable(N_mv)
    obj_mv = cp.Minimize(cp.quad_form(w_mv, Sigma_p) - (1/lam_p) * (mu_p @ w_mv))
    constraints_mv = [cp.sum(w_mv) == 1, w_mv >= 0, w_mv <= 0.60]
    prob_mv = cp.Problem(obj_mv, constraints_mv)

    # Set fixed parameters
    mu_p.value    = mu_mv
    Sigma_p.value = Sigma_mv

    # Sweep lambda to trace efficient frontier
    lambda_values = np.logspace(-1, 2, 80)
    ef_vols, ef_rets, ef_weights = [], [], []

    for lam in lambda_values:
        lam_p.value = lam
        prob_mv.solve(solver=cp.OSQP, warm_start=True, verbose=False)
        if prob_mv.status in ['optimal', 'optimal_inaccurate'] and w_mv.value is not None:
            r_p = float(w_mv.value @ mu_mv)
            v_p = float(np.sqrt(w_mv.value @ Sigma_mv @ w_mv.value))
            ef_vols.append(v_p * 100)
            ef_rets.append(r_p * 100)
            ef_weights.append(w_mv.value.copy())

    ef_vols = np.array(ef_vols)
    ef_rets = np.array(ef_rets)
    ef_weights = np.array(ef_weights)

    # Tangency portfolio: maximise Sharpe
    w_tang = cp.Variable(N_mv)
    y      = cp.Variable(N_mv, nonneg=True)   # y = w / k, k = 1/(w'μ - rf)
    # SOCP trick: max Sharpe = min vol/excess_ret = min ||Sigma^0.5 y|| s.t. (μ-rf)'y = 1
    L_mv   = np.linalg.cholesky(Sigma_mv)
    prob_tang = cp.Problem(
        cp.Minimize(cp.norm(L_mv.T @ y)),
        [(mu_mv - rf_mv) @ y == 1, cp.sum(y) >= 0]
    )
    prob_tang.solve(verbose=False)
    y_opt  = y.value
    k      = y_opt.sum()
    w_tang_opt = y_opt / k
    r_tang = float(w_tang_opt @ mu_mv)
    v_tang = float(np.sqrt(w_tang_opt @ Sigma_mv @ w_tang_opt))
    sr_tang = (r_tang - rf_mv) / v_tang

    # Min-variance portfolio
    lam_p.value = 1e6
    prob_mv.solve(solver=cp.OSQP, verbose=False)
    w_mv_opt = w_mv.value.copy()
    r_mv = float(w_mv_opt @ mu_mv)
    v_mv = float(np.sqrt(w_mv_opt @ Sigma_mv @ w_mv_opt))

    # Capital Market Line
    cml_v = np.array([0, v_tang * 1.7])
    cml_r = rf_mv * 100 + sr_tang * cml_v * 100

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Efficient frontier
    ax = axes[0]
    ax.plot(ef_vols, ef_rets, color=PURPLE, lw=2.5, label='Efficient frontier')
    ax.plot(cml_v * 100, cml_r, color=GREEN, lw=1.8, ls='--',
            label=f'Capital Market Line (slope={sr_tang:.2f})')
    ax.scatter([v_tang*100], [r_tang*100], s=130, color=AMBER, zorder=6,
               edgecolors='#BA7517', lw=1.5, label=f'Tangency (SR={sr_tang:.2f})')
    ax.scatter([v_mv*100], [r_mv*100], s=90, color='white', zorder=6,
               edgecolors=GREEN, lw=2.5, label='Min variance')
    ax.scatter(vols_mv*100, mu_mv*100, s=70, color=CORAL, zorder=5, label='Individual assets')
    for i, name in enumerate(asset_names_mv):
        ax.annotate(name, (vols_mv[i]*100, mu_mv[i]*100),
                    textcoords='offset points', xytext=(5, 2), fontsize=8, color=CORAL)
    ax.axhline(rf_mv*100, color=GRAY, lw=1, ls=':', label=f'Risk-free ({rf_mv*100:.1f}%)')
    ax.set_xlabel('Annual volatility (%)'); ax.set_ylabel('Expected return (%)')
    ax.set_title('Efficient frontier & Capital Market Line')
    ax.legend(fontsize=7)

    # Weight allocation along the frontier
    ax = axes[1]
    colors_ef = [PURPLE, CORAL, BLUE, TEAL, AMBER]
    n_show = len(ef_weights)
    for i, (name, col) in enumerate(zip(asset_names_mv, colors_ef)):
        ax.fill_between(ef_vols, 0,
                        np.cumsum(ef_weights, axis=1)[:, i] * 100 if i == 0
                        else np.cumsum(ef_weights, axis=1)[:, i] * 100,
                        label=name, color=col, alpha=0.7)
    # Actually show stacked area properly
    ax.cla()
    cumw = np.zeros(len(ef_weights))
    for i, (name, col) in enumerate(zip(asset_names_mv, colors_ef)):
        w_i = ef_weights[:, i] * 100
        ax.fill_between(ef_vols, cumw, cumw + w_i, label=name, color=col, alpha=0.75)
        cumw += w_i
    ax.axvline(v_tang*100, color=AMBER, lw=1.5, ls='--', label='Tangency vol')
    ax.axvline(v_mv*100,   color=GREEN, lw=1.5, ls=':',  label='Min-var vol')
    ax.set_xlabel('Portfolio volatility (%)'); ax.set_ylabel('Asset weight (%)')
    ax.set_title('Weight composition along the frontier\nAs lambda decreases, equity weight rises')
    ax.legend(fontsize=7, loc='upper right')

    # Risk score -> lambda -> portfolio vol mapping
    risk_scores  = np.arange(1, 11)
    lambda_map   = np.logspace(2, -0.5, 10)   # risk score 1 = most conservative
    port_vols_rs = []
    port_rets_rs = []
    for lam in lambda_map:
        lam_p.value = lam
        prob_mv.solve(solver=cp.OSQP, warm_start=True, verbose=False)
        if w_mv.value is not None:
            port_vols_rs.append(np.sqrt(w_mv.value @ Sigma_mv @ w_mv.value)*100)
            port_rets_rs.append(float(w_mv.value @ mu_mv)*100)
        else:
            port_vols_rs.append(np.nan); port_rets_rs.append(np.nan)

    ax = axes[2]
    ax2 = ax.twinx()
    ax.bar(risk_scores, port_vols_rs, color=PURPLE, alpha=0.7, width=0.6, label='Portfolio vol (%)')
    ax2.plot(risk_scores, port_rets_rs, 'o-', color=AMBER, lw=2, ms=7, label='Expected return (%)')
    ax.set_xlabel('Risk score (1=conservative, 10=aggressive)')
    ax.set_ylabel('Annual volatility (%)', color=PURPLE)
    ax2.set_ylabel('Expected return (%)', color=AMBER)
    ax.set_title('Risk score -> lambda -> portfolio\n(how a robo-advisor maps client risk preferences)')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f'Tangency portfolio (max Sharpe = {sr_tang:.3f}):')
    for name, w_v in zip(asset_names_mv, w_tang_opt):
        print(f'  {name:15s}: {w_v*100:.1f}%')
    print(f'  Return: {r_tang*100:.1f}%  |  Vol: {v_tang*100:.1f}%  |  Sharpe: {sr_tang:.3f}')

---
## Part 6 — Rebalancing as QP+L1: Full Problem Formulation

The rebalancing problem is not just minimum variance — it must balance three competing objectives simultaneously.

### The complete rebalancing QP
$$\underset{\mathbf{w}}{\text{minimise}} \quad \underbrace{\lambda_{TE}\,(\mathbf{w}-\mathbf{w}^*)^\top\Sigma(\mathbf{w}-\mathbf{w}^*)}_{\text{tracking error variance}} + \underbrace{\lambda_{TC}\,\mathbf{c}^\top|\mathbf{w}-\mathbf{w}_{curr}|}_{\text{transaction costs (L1)}} + \underbrace{\lambda_{tax}\sum_i \tau_i\max(0, w_{curr,i}-w_i)\cdot g_i}_{\text{tax on realized gains}}$$

$$\text{s.t.}\quad \mathbf{1}^\top\mathbf{w}=1,\; \mathbf{w}\geq 0,\; w_i \leq c_i,\; \|\Delta\mathbf{w}\|_1 \leq T_{max}$$

### Why L1 costs make this a QP (not just LP)
The transaction cost term $\mathbf{c}^\top|\Delta\mathbf{w}|$ is piecewise-linear and convex. Combined with the quadratic tracking error term, the problem is a **QP with L1 regularization** — exactly the form that OSQP was designed for.

The standard trick to make $|\Delta w_i|$ solver-friendly: introduce auxiliary variables $t_i \geq 0$ with:
$$t_i \geq w_i - w_{curr,i} \quad \text{and} \quad t_i \geq w_{curr,i} - w_i$$

At optimality, $t_i = |w_i - w_{curr,i}|$. cvxpy's `cp.abs()` does this transformation automatically.

### No-trade zone
The optimal solution is $\mathbf{w}^* = \mathbf{w}_{curr}$ (don't trade) when the tracking error penalty is small relative to transaction costs. The **no-trade zone** is the polyhedron in weight-space where:
$$2\lambda_{TE}\|(\Sigma(\mathbf{w}_{curr}-\mathbf{w}^*))_i\| \leq \lambda_{TC}\,c_i \quad \forall i$$
Portfolios inside this zone have insufficient tracking error to justify the transaction cost.

In [ ]:
# ── Rebalancing optimizer: full QP+L1 with parameter sweep ──────────────────
if not CVXPY:
    print('cvxpy required')
else:
    N_reb  = 4
    names_reb = ['MSCI World', 'EM Equity', 'Gov Bond', 'Corp Bond']
    vols_r = np.array([0.18, 0.22, 0.06, 0.10])
    corr_r = np.array([[ 1.00,  0.75, -0.20,  0.20],
                        [ 0.75,  1.00, -0.15,  0.15],
                        [-0.20, -0.15,  1.00,  0.60],
                        [ 0.20,  0.15,  0.60,  1.00]])
    Sigma_r = np.outer(vols_r, vols_r) * corr_r

    w_curr_r = np.array([0.52, 0.18, 0.10, 0.20])   # drifted current weights
    w_tgt_r  = np.array([0.35, 0.20, 0.25, 0.20])   # target weights
    cost_r2  = np.array([0.0003, 0.0008, 0.0002, 0.0004])
    V_r      = 100_000.0
    TE_PRE   = float(np.sqrt((w_curr_r - w_tgt_r) @ Sigma_r @ (w_curr_r - w_tgt_r))) * 100

    # ── Template with Parameters (compiled once) ─────────────────────────────
    Sp_r    = cp.Parameter((N_reb, N_reb), PSD=True)
    wcp_r   = cp.Parameter(N_reb)
    wtp_r   = cp.Parameter(N_reb)
    cosp_r  = cp.Parameter(N_reb, nonneg=True)
    lte_r   = cp.Parameter(nonneg=True)
    ltc_r   = cp.Parameter(nonneg=True)

    w_r = cp.Variable(N_reb)
    delta = w_r - wcp_r

    obj_r    = cp.Minimize(lte_r * cp.quad_form(w_r - wtp_r, Sp_r) + ltc_r * (cosp_r @ cp.abs(delta)))
    constr_r = [cp.sum(w_r)==1, w_r>=0, w_r<=0.55, cp.sum(cp.abs(delta))<=0.30]
    prob_r   = cp.Problem(obj_r, constr_r)

    # Set fixed params
    Sp_r.value   = Sigma_r
    wcp_r.value  = w_curr_r
    wtp_r.value  = w_tgt_r
    cosp_r.value = cost_r2

    # Sweep lambda_TE (higher = trade more aggressively)
    lam_te_vals = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
    ltc_r.value = 1.0
    sweep_res = []
    for lte in lam_te_vals:
        lte_r.value = lte
        prob_r.solve(solver=cp.OSQP, warm_start=True, verbose=False)
        if w_r.value is not None:
            w_opt_r = w_r.value.copy()
            te  = float(np.sqrt((w_opt_r - w_tgt_r) @ Sigma_r @ (w_opt_r - w_tgt_r))) * 100
            tc  = float(cost_r2 @ np.abs(w_opt_r - w_curr_r)) * 100
            trn = float(np.abs(w_opt_r - w_curr_r).sum()) * 100
            sweep_res.append({'lte': lte, 'w': w_opt_r, 'te': te, 'tc': tc, 'turnover': trn})

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Weight evolution as lambda_TE increases
    ax = axes[0]
    lte_arr = [r['lte'] for r in sweep_res]
    colors_a = [PURPLE, CORAL, BLUE, TEAL]
    for i, (name, col) in enumerate(zip(names_reb, colors_a)):
        ws = [r['w'][i]*100 for r in sweep_res]
        ax.plot(lte_arr, ws, 'o-', color=col, lw=2, ms=6, label=name)
        ax.axhline(w_tgt_r[i]*100, color=col, lw=1, ls='--', alpha=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('TE penalty lambda_TE (log scale)')
    ax.set_ylabel('Optimal weight (%)')
    ax.set_title('Higher lambda_TE -> trade more aggressively\nDashed = target weights')
    ax.legend(fontsize=8)
    ax.text(0.02, 0.02, 'Dashed = target', transform=ax.transAxes, fontsize=8, color=GRAY)

    # TE vs TC trade-off curve
    ax = axes[1]
    te_v = [r['te'] for r in sweep_res]
    tc_v = [r['tc'] for r in sweep_res]
    ax.plot(tc_v, te_v, 'o-', color=PURPLE, lw=2.5, ms=9)
    for i, r in enumerate(sweep_res):
        ax.annotate(f'lambda={r["lte"]:.0f}', (tc_v[i], te_v[i]),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)
    ax.scatter([tc_v[0]],  [te_v[0]],  s=130, color=CORAL,  zorder=6, label='Low lambda (stay put)')
    ax.scatter([tc_v[-1]], [te_v[-1]], s=130, color=GREEN, zorder=6, label='High lambda (full rebal)')
    ax.axhline(TE_PRE, color=GRAY, lw=1.5, ls=':', label=f'Pre-trade TE={TE_PRE:.2f}%')
    ax.set_xlabel('Transaction cost (% of portfolio)')
    ax.set_ylabel('Post-trade tracking error (%)')
    ax.set_title('Cost vs TE trade-off frontier\n(Pareto-optimal rebalancing decisions)')
    ax.legend(fontsize=8)

    # Optimal trade list (high lambda)
    ax = axes[2]
    best_r = sweep_res[-1]
    trades_r = (best_r['w'] - w_curr_r) * 100
    trade_eur = trades_r / 100 * V_r
    cols_t = [GREEN if t > 0 else RED for t in trades_r]
    bars_t = ax.barh(names_reb, trades_r, color=cols_t, edgecolor='white', height=0.55)
    ax.axvline(0, color='black', lw=1)
    for bar, t, e in zip(bars_t, trades_r, trade_eur):
        x_pos = bar.get_width()
        ax.text(x_pos + (0.1 if x_pos >= 0 else -0.1), bar.get_y()+bar.get_height()/2,
                f'{t:+.1f}pp / €{e:+,.0f}',
                va='center', ha='left' if x_pos >= 0 else 'right', fontsize=9)
    ax.set_xlabel('Trade size (percentage points)')
    ax.set_title(f'Optimal trade list (lambda_TE={lte_arr[-1]:.0f})\nV=€{V_r:,.0f}')

    plt.tight_layout()
    plt.show()

    print('Rebalancing results (lambda_TC=1.0):')
    print(f'  Pre-trade TE: {TE_PRE:.3f}%')
    print(f'  {"lambda_TE":>10} | {"Post TE":>8} | {"TC cost":>8} | {"Turnover":>10}')
    print('  ' + '-'*44)
    for r in sweep_res:
        print(f'  {r["lte"]:>10.1f} | {r["te"]:>7.3f}% | {r["tc"]:>7.4f}% | {r["turnover"]:>9.2f}%')

---
## Part 7 — CVaR Minimization as a Linear Program (Rockafellar-Uryasev 2000)

CVaR appears to be a complex non-linear functional of the portfolio. Rockafellar and Uryasev (2000) proved it can be computed and minimized exactly as a **Linear Program** over discrete scenarios. This is one of the most important results in computational finance.

### The theorem
For $T$ scenarios with return matrix $R$ (shape $T \times N$) and portfolio weights $\mathbf{w}$:
$$\text{CVaR}_\alpha(\mathbf{w}) = \min_{\beta \in \mathbb{R}} \left\{\beta + \frac{1}{(1-\alpha)T}\sum_{t=1}^T \max\!\left(-\mathbf{r}_t^\top\mathbf{w} - \beta,\; 0\right)\right\}$$

At the optimum, $\beta^* = \text{VaR}_\alpha(\mathbf{w})$. Introducing auxiliary variables $z_t = \max(-\mathbf{r}_t^\top\mathbf{w} - \beta, 0)$:

$$\text{CVaR}_\alpha(\mathbf{w}) = \min_{\mathbf{w}, \beta, \mathbf{z}} \quad \beta + \frac{1}{(1-\alpha)T}\mathbf{1}^\top\mathbf{z}$$

$$\text{s.t.}\quad z_t \geq -R_{t:}\mathbf{w} - \beta \quad \forall t, \quad z_t \geq 0 \quad \forall t$$

This is a **linear program** in $(\mathbf{w}, \beta, \mathbf{z})$ with $N + 1 + T$ variables and $2T$ inequality constraints.

### Problem size and speed
For $N=50$ assets, $T=1000$ scenarios, $\alpha=0.95$:
- Variables: $50 + 1 + 1000 = 1051$
- Constraints: $2000 + $ portfolio constraints

ECOS solves this in <50ms. This makes CVaR minimization **entirely practical** for production — you can run it per-client on every rebalancing cycle.

### Combined CVaR + tracking error objective
In production, you'd combine CVaR with tracking error:
$$\min_{\mathbf{w}} \quad \lambda_{CVaR} \cdot \text{CVaR}_\alpha(\mathbf{w}) + \lambda_{TE} \cdot (\mathbf{w}-\mathbf{w}^*)^\top\Sigma(\mathbf{w}-\mathbf{w}^*)$$

This is a **SOCP** (because CVaR is linear and TE is quadratic). cvxpy handles this transparently.

In [ ]:
# ── CVaR minimization: Rockafellar-Uryasev LP ───────────────────────────────
if not CVXPY:
    print('cvxpy required')
else:
    np.random.seed(42)
    N_cv   = 4
    T_cv   = 600    # 600 historical / simulated scenarios
    alpha  = 0.95
    names_cv = ['MSCI World', 'EM Equity', 'Gov Bond', 'Corp Bond']

    # Scenario return matrix: fat-tailed (Student-t df=5)
    mu_d    = np.array([0.0004, 0.0005, 0.0001, 0.0003])
    vols_d  = np.array([0.012, 0.015, 0.004, 0.009])
    corr_d  = np.array([[ 1.0,  0.70, -0.20,  0.35],
                         [ 0.70,  1.0, -0.10,  0.25],
                         [-0.20, -0.10,  1.0,  0.55],
                         [ 0.35,  0.25,  0.55,  1.0]])
    Sig_d   = np.outer(vols_d, vols_d) * corr_d
    L_d     = np.linalg.cholesky(Sig_d)
    z_sc    = t_dist.rvs(df=5, size=(T_cv, N_cv)) * np.sqrt(3.0/5.0)   # scale to unit variance
    R_sc    = z_sc @ L_d.T + mu_d   # (T, N) scenario matrix

    # ── Problem 1: Minimize CVaR (Rockafellar-Uryasev LP) ─────────────────
    w_cv   = cp.Variable(N_cv)
    beta   = cp.Variable()
    z_aux  = cp.Variable(T_cv)
    port_rets_cv = R_sc @ w_cv   # (T,) portfolio return per scenario

    cvar_objective = beta + (1.0 / ((1 - alpha) * T_cv)) * cp.sum(z_aux)
    prob_cvar2 = cp.Problem(
        cp.Minimize(cvar_objective),
        [z_aux >= 0,
         z_aux >= -port_rets_cv - beta,
         cp.sum(w_cv) == 1,
         w_cv >= 0,
         w_cv <= 0.60,
         w_cv @ mu_d >= 0.0003]
    )
    prob_cvar2.solve(solver=cp.ECOS, verbose=False)
    w_cvar_opt = w_cv.value.copy()
    var_opt    = float(beta.value)

    # ── Problem 2: Min variance for comparison ─────────────────────────────
    w_minv = cp.Variable(N_cv)
    prob_mv_cv = cp.Problem(
        cp.Minimize(cp.quad_form(w_minv, Sig_d)),
        [cp.sum(w_minv)==1, w_minv>=0, w_minv<=0.60]
    )
    prob_mv_cv.solve(verbose=False)
    w_mv_cv = w_minv.value.copy()

    # ── Problem 3: Equal-weight portfolio for baseline ─────────────────────
    w_eq = np.ones(N_cv) / N_cv

    # ── Compute risk metrics for each portfolio ────────────────────────────
    def risk_metrics(w, R, alpha_v=0.95):
        pr = R @ w
        sorted_pr = np.sort(pr)
        idx = int((1 - alpha_v) * len(pr))
        var_v  = -sorted_pr[idx]
        cvar_v = -sorted_pr[:idx].mean()
        vol_v  = pr.std() * np.sqrt(252) * 100   # annualised
        ret_v  = pr.mean() * 252 * 100
        return {'VaR': var_v*100, 'CVaR': cvar_v*100, 'vol_ann': vol_v, 'ret_ann': ret_v}

    metrics = {
        'Min CVaR':       risk_metrics(w_cvar_opt, R_sc),
        'Min variance':   risk_metrics(w_mv_cv, R_sc),
        'Equal weight':   risk_metrics(w_eq, R_sc),
    }

    # ── Plot ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Weight comparison
    x_cv = np.arange(N_cv); w_cv_b = 0.25
    colors_cv3 = [TEAL, PURPLE, GRAY]
    labels_cv3 = ['Min CVaR (LP)', 'Min variance (QP)', 'Equal weight']
    portfolios_cv = [w_cvar_opt, w_mv_cv, w_eq]
    for i, (w_p, col, lab) in enumerate(zip(portfolios_cv, colors_cv3, labels_cv3)):
        axes[0].bar(x_cv + (i-1)*w_cv_b, w_p*100, w_cv_b,
                    color=col, label=lab, alpha=0.85, edgecolor='white')
    axes[0].set_xticks(x_cv); axes[0].set_xticklabels(names_cv, rotation=15, fontsize=9)
    axes[0].set_ylabel('Weight (%)')
    axes[0].set_title('Min-CVaR vs min-variance vs equal-weight\nDifferent tail-risk objectives -> different allocations')
    axes[0].legend(fontsize=8)

    # Return distribution tails
    ax = axes[1]
    for w_p, col, lab in zip(portfolios_cv, colors_cv3, labels_cv3):
        pr = R_sc @ w_p
        m  = metrics[lab]
        ax.hist(pr*100, bins=50, color=col, alpha=0.4, density=True, label=f'{lab} (CVaR={m["CVaR"]:.2f}%)')
        ax.axvline(-m['CVaR'], color=col, lw=2, ls='--')
    ax.set_xlabel('Daily portfolio return (%)')
    ax.set_ylabel('Density')
    ax.set_title('Return distributions: fat tails\nDashed = CVaR95 for each portfolio')
    ax.legend(fontsize=7)

    # CVaR as function of alpha
    alphas_v = np.linspace(0.90, 0.995, 25)
    for w_p, col, lab in zip(portfolios_cv, colors_cv3, labels_cv3):
        cvar_arr = [risk_metrics(w_p, R_sc, a)['CVaR'] for a in alphas_v]
        axes[2].plot(alphas_v*100, cvar_arr, color=col, lw=2, label=lab)
    axes[2].set_xlabel('Confidence level alpha (%)')
    axes[2].set_ylabel('CVaR (% daily loss)')
    axes[2].set_title('CVaR across confidence levels\nMin-CVaR portfolio dominates in the tail')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f'Portfolio comparison ({T_cv} scenarios, alpha={alpha}):')
    print(f'  {"Portfolio":<18} {"Return":>9} {"Vol(ann)":>10} {"VaR95":>9} {"CVaR95":>9}')
    print('  ' + '-'*58)
    for lab, m in metrics.items():
        print(f'  {lab:<18} {m["ret_ann"]:>8.1f}% {m["vol_ann"]:>9.1f}% {m["VaR"]:>8.3f}% {m["CVaR"]:>8.3f}%')

    print(f'\nLP problem size (Rockafellar-Uryasev):')
    print(f'  Variables: N + 1 + T = {N_cv} + 1 + {T_cv} = {N_cv + 1 + T_cv}')
    print(f'  Constraints: 2T + portfolio = {2*T_cv} + {N_cv + 2} = {2*T_cv + N_cv + 2}')
    print(f'  Solver: ECOS | Status: {prob_cvar2.status}')

---
## Part 8 — Production Patterns: Building a Robust Optimizer

Running convex optimization in production for hundreds of thousands of clients requires careful engineering. Here is the complete production template.

### Design principles
1. **Compile the problem template once at module load** — the graph compilation costs ~5–20ms; the solve costs ~1–50ms. Never recompile per-client.
2. **Use `cp.Parameter` for everything client-specific** — Sigma, current weights, target weights, cost rates, tax rates
3. **Always check `prob.status`** before using the solution — status can be `'optimal'`, `'optimal_inaccurate'`, `'infeasible'`, `'unbounded'`, or `'solver_error'`
4. **Implement a fallback** — if the optimizer fails, fall back to proportional rebalancing (trade toward target proportionally) rather than doing nothing
5. **Log solve time and status** — production monitoring must catch solver degradation
6. **Idempotency** — the same job must produce the same result if run twice (at-least-once delivery from SQS queues)

### Common failure modes and fixes

| Failure | Cause | Fix |
|---|---|---|
| `'infeasible'` | Constraints contradict (e.g., required return > achievable) | Relax binding constraint; add slack variable |
| `'unbounded'` | Objective goes to $-\infty$ | Missing constraint; check problem structure |
| `'optimal_inaccurate'` | Hit iteration limit | Increase `max_iter`; try different solver |
| Numerical instability | Poorly scaled inputs | Normalise weights to [0,1]; covariance in consistent units |
| Slow warm-start | Very different consecutive problems | Disable warm-start; reset to default initialization |

### The `cp.Parameter` PSD declaration
When you declare `cp.Parameter((N,N), PSD=True)`, cvxpy trusts that the assigned matrix is positive semi-definite. It does NOT check. You must ensure it — use `sklearn.covariance.LedoitWolf` or add a small diagonal: `Sigma + eps * np.eye(N)`. If you pass a non-PSD matrix, the solver may return garbage or `'solver_error'`.

In [ ]:
# ── Production-grade rebalancer: Parameters, fallback, status handling ───────
if not CVXPY:
    print('cvxpy required')
else:
    class ProductionRebalancer:
        """
        Production-grade rebalancing optimizer.
        - Compiles once at init time.
        - Updates cp.Parameters per solve (no recompilation).
        - Warm starts from previous solution.
        - Full status checking and fallback.
        """
        def __init__(self, N: int, max_weight: float = 0.55, max_turnover: float = 0.30):
            self.N = N
            self.n_solves = 0
            self.n_fallbacks = 0
            self._build(N, max_weight, max_turnover)

        def _build(self, N, max_weight, max_turnover):
            # ── Parameters (set per client, no recompilation) ─────────────
            self.Sigma_p   = cp.Parameter((N, N), PSD=True,   name='Sigma')
            self.w_curr_p  = cp.Parameter(N,                  name='w_curr')
            self.w_tgt_p   = cp.Parameter(N,                  name='w_tgt')
            self.cost_p    = cp.Parameter(N, nonneg=True,     name='costs')
            self.lam_te_p  = cp.Parameter(nonneg=True,        name='lam_te')
            self.lam_tc_p  = cp.Parameter(nonneg=True,        name='lam_tc')

            # ── Decision variable ─────────────────────────────────────────
            self.w = cp.Variable(N, name='weights')
            delta  = self.w - self.w_curr_p

            # ── Objective ────────────────────────────────────────────────
            te_term  = self.lam_te_p * cp.quad_form(self.w - self.w_tgt_p, self.Sigma_p)
            tc_term  = self.lam_tc_p * (self.cost_p @ cp.abs(delta))
            objective = cp.Minimize(te_term + tc_term)

            # ── Constraints ───────────────────────────────────────────────
            self.constraints = [
                cp.sum(self.w) == 1,
                self.w >= 0,
                self.w <= max_weight,
                cp.sum(cp.abs(delta)) <= max_turnover,
            ]

            # ── Compile once ──────────────────────────────────────────────
            self.prob = cp.Problem(objective, self.constraints)
            print(f'Problem compiled: {self.prob.is_dcp()} DCP, '
                  f'{len(self.prob.variables())} vars, '
                  f'{len(self.prob.constraints)} constraints')

        def solve(self, Sigma, w_curr, w_tgt, cost_rates,
                  lam_te=10.0, lam_tc=1.0, eps_psd=1e-6) -> dict:
            """Solve for one client. O(solve_time) — no recompilation."""
            t0 = time.time()
            self.n_solves += 1

            # Ensure PSD (add small diagonal regularization)
            Sigma_reg = Sigma + eps_psd * np.eye(self.N)

            # Update parameters
            self.Sigma_p.value  = Sigma_reg
            self.w_curr_p.value = w_curr
            self.w_tgt_p.value  = w_tgt
            self.cost_p.value   = cost_rates
            self.lam_te_p.value = lam_te
            self.lam_tc_p.value = lam_tc

            self.prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)
            dt = time.time() - t0

            status = self.prob.status
            w_opt  = self.w.value

            if status in ('optimal', 'optimal_inaccurate') and w_opt is not None:
                w_opt = np.clip(w_opt, 0, 1)
                w_opt /= w_opt.sum()   # renormalize for numerical cleanliness
                method = 'optimizer'
            else:
                # Fallback: proportional partial rebalancing
                self.n_fallbacks += 1
                rebal_fraction = 0.50   # move 50% of the way toward target
                w_opt = w_curr + rebal_fraction * (w_tgt - w_curr)
                w_opt = np.clip(w_opt, 0, 1)
                w_opt /= w_opt.sum()
                method = f'fallback (status={status})'

            trades = w_opt - w_curr
            te_post = float(np.sqrt((w_opt - w_tgt) @ Sigma @ (w_opt - w_tgt))) * 100
            tc_est  = float(cost_rates @ np.abs(trades)) * 100

            return {
                'w_optimal': w_opt,
                'trades': trades,
                'te_post': te_post,
                'tc_est': tc_est,
                'status': status,
                'method': method,
                'solve_ms': dt * 1000,
            }

        def stats(self):
            return f'Solves: {self.n_solves}, Fallbacks: {self.n_fallbacks} ({self.n_fallbacks/max(1,self.n_solves)*100:.1f}%)'

    # ── Benchmark: solve for 200 simulated clients ───────────────────────────
    rebalancer = ProductionRebalancer(N=N_reb, max_weight=0.55, max_turnover=0.30)

    n_clients = 200
    np.random.seed(42)
    solve_times = []
    te_values   = []
    tc_values   = []
    statuses    = []

    for i in range(n_clients):
        # Simulate random client state
        noise = np.random.normal(0, 0.04, N_reb)
        wc = np.clip(w_tgt_r + noise, 0, 1); wc /= wc.sum()
        res = rebalancer.solve(Sigma_r, wc, w_tgt_r, cost_r2, lam_te=10.0, lam_tc=1.0)
        solve_times.append(res['solve_ms'])
        te_values.append(res['te_post'])
        tc_values.append(res['tc_est'])
        statuses.append(res['status'])

    # ── Visualise benchmark ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    axes[0].hist(solve_times, bins=30, color=PURPLE, alpha=0.8, edgecolor='white')
    axes[0].axvline(np.mean(solve_times), color=AMBER, lw=2, ls='--',
                    label=f'Mean: {np.mean(solve_times):.1f} ms')
    axes[0].axvline(np.percentile(solve_times, 95), color=RED, lw=2, ls='--',
                    label=f'P95: {np.percentile(solve_times, 95):.1f} ms')
    axes[0].set_xlabel('Solve time (ms)')
    axes[0].set_ylabel('Number of clients')
    axes[0].set_title(f'Solve time distribution\n({n_clients} clients, N={N_reb} assets, warm-start OSQP)')
    axes[0].legend(fontsize=9)

    axes[1].scatter(te_values, tc_values, c=solve_times, cmap='viridis', s=20, alpha=0.7)
    axes[1].set_xlabel('Post-trade tracking error (%)')
    axes[1].set_ylabel('Transaction cost (%)')
    axes[1].set_title('TE vs cost across clients\n(color = solve time ms)')
    sm = plt.cm.ScalarMappable(cmap='viridis',
                                norm=plt.Normalize(min(solve_times), max(solve_times)))
    plt.colorbar(sm, ax=axes[1], shrink=0.8, label='Solve time (ms)')

    unique_statuses = {}
    for s in statuses:
        unique_statuses[s] = unique_statuses.get(s, 0) + 1
    axes[2].bar(unique_statuses.keys(), unique_statuses.values(),
                color=[GREEN if k=='optimal' else AMBER if k=='optimal_inaccurate' else RED
                       for k in unique_statuses.keys()],
                edgecolor='white', width=0.5)
    axes[2].set_ylabel('Number of clients')
    axes[2].set_title(f'Solver status distribution\n{rebalancer.stats()}')
    for i, (k, v) in enumerate(unique_statuses.items()):
        axes[2].text(i, v + 0.5, str(v), ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'Benchmark summary ({n_clients} clients):')
    print(f'  Mean solve time:  {np.mean(solve_times):.2f} ms')
    print(f'  P50 solve time:   {np.percentile(solve_times, 50):.2f} ms')
    print(f'  P95 solve time:   {np.percentile(solve_times, 95):.2f} ms')
    print(f'  P99 solve time:   {np.percentile(solve_times, 99):.2f} ms')
    print(f'  Estimated 300k clients: {np.mean(solve_times)*300000/1000/60:.1f} CPU-minutes total')
    print(f'  At 1000 Lambda workers: {np.mean(solve_times)*300000/1000/1000:.0f} seconds wall time')
    print(f'  {rebalancer.stats()}')

---
## Summary: Convex Optimization Mental Map

| Problem class | Finance application | cvxpy expression | Solver |
|---|---|---|---|
| **LP** | CVaR minimisation, index replication | `cp.Minimize(c @ x)` | ECOS, HiGHS |
| **QP** | Min-variance, risk parity | `cp.Minimize(cp.quad_form(w, Sigma))` | OSQP, Clarabel |
| **QP + L1** | Rebalancing with transaction costs | `+ cost @ cp.abs(delta)` | OSQP |
| **QCQP / SOCP** | Max-return with vol cap, robust MVO | `cp.quad_form(w, Sigma) <= sigma_max**2` | ECOS, Clarabel |
| **CVaR LP** | Tail-risk optimisation | `beta + (1/((1-alpha)*T)) * cp.sum(z)` | ECOS |

### The five things to say in any interview about convex optimization

1. **Convexity guarantees a unique global minimum** — every portfolio optimization problem should be formulated as convex by design; if it isn't, you're fighting the solver

2. **The problem hierarchy LP → QP → SOCP → SDP** — know what problem class each finance objective belongs to, and which solver to use

3. **KKT dual variables have economic meaning** — the shadow price of the budget constraint is the marginal value of capital; the shadow price of a volatility constraint is the cost of additional risk

4. **CVaR is a convex LP (Rockafellar-Uryasev)** — not an exotic computation; it can be directly minimised over scenarios as a linear program

5. **Use `cp.Parameter` + `warm_start=True` in production** — compile once, solve many times; 100x speedup vs recompilation; mandatory for 300K client scale

### Key equations
$$\sigma^2_p = \mathbf{w}^\top\Sigma\mathbf{w} \quad \text{(QP objective, convex because } \Sigma \succeq 0\text{)}$$
$$\text{CVaR}_\alpha = \min_{\beta} \left\{ \beta + \frac{1}{(1-\alpha)T}\sum_t \max(-\mathbf{r}_t^\top\mathbf{w}-\beta, 0)\right\} \quad \text{(LP in } \mathbf{w}, \beta, \mathbf{z}\text{)}$$
$$\lambda_i f_i(x^*) = 0 \quad \forall i \quad \text{(complementary slackness — binding constraint or zero shadow price)}$$